# MODELO DE RED NEURONAL

## 1. Instalación de librerías

In [3]:
import pandas as pd
import numpy as np
import re
import nltk
import spacy
import warnings
warnings.filterwarnings("ignore")

from nltk.corpus import stopwords

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

# TensorFlow / Keras
import tensorflow as tf
import keras
from keras import layers, regularizers
from keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
)

In [4]:
# Descargar recursos de NLTK
nltk.download("stopwords")
nltk.download("punkt")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dacma\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dacma\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [5]:
# Cargar modelo de spaCy en español
def cargar_modelo_spacy_es():
    for model_name in ("es_core_news_md", "es_core_news_sm"):
        try:
            print(f"Cargando modelo spaCy: {model_name}")
            return spacy.load(model_name)
        except OSError:
            continue

    print("No se encontro un modelo de spaCy en espanol instalado. Se usara un pipeline basico.")
    return spacy.blank("es")


nlp = cargar_modelo_spacy_es()

print("✅ Librerías cargadas correctamente.")

Cargando modelo spaCy: es_core_news_md
✅ Librerías cargadas correctamente.


## 2. CARGA DE DATOS

In [6]:
df = pd.read_excel("data/Data_arreglada.xlsx", sheet_name="Sheet1")

In [7]:
# Renombrar columnas para facilitar el trabajo
df.columns = ["Categoria", "Resuelta", "Consulta"]

print(f"📦 Shape original: {df.shape}")
print(df.head(5))

📦 Shape original: (36468, 3)
                                   Categoria Resuelta  \
0                              Carnetización       No   
1                       Calendario académico       No   
2                              Carnetización       Sí   
3  ​ Gestión Económica - Información general       Sí   
4                              Carnetización       No   

                                            Consulta  
0  Ayuda no se como inscribir asignaturas y nadie...  
1           Donde puedo ver el calendario académico?  
2                                                NaN  
3                                                NaN  
4                              Quiero mi carné nuevo  


## 3. LIMPIEZA INICIAL 

In [12]:
# Eliminar filas donde Consulta o Categoria sean nulas
df = df.dropna(subset=["Consulta", "Categoria"])

In [13]:
# Eliminar filas que no fueron resueltas (Resuelta = 'Sí')
df = df[df["Resuelta"].str.strip().str.lower() != "sí"]
print(f"\n📦 Shape después de eliminar resueltas: {df.shape}")


📦 Shape después de eliminar resueltas: (35168, 3)


In [14]:
# Eliminar filas con consulta vacía o solo espacios
df = df[df["Consulta"].str.strip() != ""]
df = df.reset_index(drop=True)

In [15]:
print(f" Shape final del dataset: {df.shape}")
print(df.head(15))

 Shape final del dataset: (35168, 3)
                                            Categoria Resuelta  \
0                                       Carnetización       No   
1                                Calendario académico       No   
2                                       Carnetización       No   
3                                Calendario académico       No   
4                                Calendario académico       No   
5           ​ Gestión Económica - Información general       No   
6                                Calendario académico       No   
7           ​ Gestión Económica - Información general       No   
8                                Calendario académico       No   
9                                       Carnetización       No   
10                               Calendario académico       No   
11                                      Carnetización       No   
12          ​ Gestión Económica - Información general       No   
13  ​ Gestión Económica - Inconsistenci

## 4. MAPEO DE CATEGORÍAS VÁLIDAS

In [16]:
CATEGORIAS_VALIDAS = [
    "Carnetización",
    "Actualización de datos personales",
    "Calendario académico",
    "Certificados",
    "Gestión Académica",
    "Gestión Económica",
    "Reubicación socioeconómica en Pregrado",
    "Aplazamiento de matrícula inicial",
    "Política de gratuidad (matrícula cero) Pregrado",
    "Información general sobre servicios estudiantiles",
]

def mapear_categoria(categoria: str) -> str:
    """
    Mapea cada categoría original del dataset a una de las CATEGORIAS_VALIDAS.
    Usa coincidencia parcial por palabras clave.
    """
    categoria = str(categoria).strip()

    # Mapeos directos y por palabras clave
    mapeo = {
        "Carnetización":                                    ["carnetización", "carnet", "carné"],
        "Actualización de datos personales":                ["actualización de datos", "datos personales", "actualización"],
        "Calendario académico":                             ["calendario académico", "calendario"],
        "Certificados":                                     ["certificado"],
        "Gestión Académica":                                ["gestión académica", "inscripción", "adiciones",
                                                             "cancelaciones", "asignaturas", "sobrecupo",
                                                             "historia académica", "bloqueo", "grado",
                                                             "homologación", "traslado", "aplazamiento",
                                                             "reingreso", "notas", "prueba", "inglés",
                                                             "doble titulación", "posgrado"],
        "Gestión Económica":                                ["gestión económica", "recibo", "pago",
                                                             "fraccionamiento", "unificación", "devolución",
                                                             "financiación", "matrícula", "pbm",
                                                             "descuento", "electoral", "generación e",
                                                             "icetex", "ser pilo", "exención",
                                                             "reexpedición", "cobro", "deuda"],
        "Reubicación socioeconómica en Pregrado":           ["reubicación socioeconómica", "reubicación",
                                                             "socioeconómica", "socioeconómico"],
        "Aplazamiento de matrícula inicial":                ["aplazamiento de matrícula", "aplazamiento inicial",
                                                             "aplazamiento"],
        "Política de gratuidad (matrícula cero) Pregrado": ["matrícula cero", "gratuidad", "matrícula 0",
                                                             "política de gratuidad"],
        "Información general sobre servicios estudiantiles":["información general", "información financiera",
                                                              "servicios estudiantiles", "bienestar",
                                                              "alimentaria", "correo institucional",
                                                              "sia", "bicirún", "sibu"],
    }

    categoria_lower = categoria.lower()

    for cat_valida, palabras_clave in mapeo.items():
        for palabra in palabras_clave:
            if palabra in categoria_lower:
                return cat_valida

    # Si no hay coincidencia, intentar por la consulta (fallback)
    return "Información general sobre servicios estudiantiles"


# Aplicar el mapeo
df["Categoria_Mapeada"] = df["Categoria"].apply(mapear_categoria)

print("\n📊 Distribución de categorías mapeadas:")
print(df["Categoria_Mapeada"].value_counts())


📊 Distribución de categorías mapeadas:
Categoria_Mapeada
Gestión Económica                                    15751
Gestión Académica                                    10520
Información general sobre servicios estudiantiles     4714
Certificados                                          2656
Actualización de datos personales                      768
Carnetización                                          458
Calendario académico                                   232
Reubicación socioeconómica en Pregrado                  49
Política de gratuidad (matrícula cero) Pregrado         20
Name: count, dtype: int64


## 5. PREPROCESAMIENTO DE TEXTO

In [17]:
STOPWORDS_ES = set(stopwords.words("spanish"))

# Stopwords adicionales específicas del dominio universitario
STOPWORDS_EXTRA = {
    "universidad", "nacional", "colombia", "unal", "sede", "bogotá",
    "favor", "gracias", "buenas", "buenos", "días", "tardes", "noches",
    "cordial", "saludo", "atentamente", "amablemente", "presente",
    "correo", "motivo", "solicito", "solicitud", "quisiera", "quiero",
    "necesito", "requiero", "agradezco", "agradecería", "muchas",
    "manera", "forma", "caso", "parte", "vez", "día", "semestre",
    "periodo", "académico", "estudiante", "programa", "curricular",
    "sia", "dninfoa", "portal", "plataforma", "sistema",
}

STOPWORDS_COMPLETO = STOPWORDS_ES.union(STOPWORDS_EXTRA)


def limpiar_texto(texto: str) -> str:
    """Limpieza básica: minúsculas, eliminar caracteres especiales y números."""
    texto = str(texto).lower()
    # Eliminar URLs
    texto = re.sub(r"http\S+|www\S+", " ", texto)
    # Eliminar correos electrónicos
    texto = re.sub(r"\S+@\S+", " ", texto)
    # Eliminar números y caracteres especiales, conservar letras y espacios
    texto = re.sub(r"[^a-záéíóúüñ\s]", " ", texto)
    # Eliminar espacios múltiples
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def eliminar_stopwords(texto: str) -> str:
    """Elimina stopwords del texto."""
    tokens = texto.split()
    tokens_filtrados = [t for t in tokens if t not in STOPWORDS_COMPLETO and len(t) > 2]
    return " ".join(tokens_filtrados)


def lematizar(texto: str) -> str:
    """Lematiza el texto usando spaCy."""
    doc = nlp(texto)
    lemas = [
        token.lemma_ if token.lemma_ else token.text
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and len(token.lemma_ if token.lemma_ else token.text) > 2
    ]
    return " ".join(lemas)


def preprocesar(texto: str) -> str:
    """Pipeline completo: limpieza → stopwords → lematización."""
    texto = limpiar_texto(texto)
    texto = eliminar_stopwords(texto)
    texto = lematizar(texto)
    return texto


print("\n⚙️  Aplicando preprocesamiento (puede tardar unos minutos)...")
df["Consulta_Procesada"] = df["Consulta"].apply(preprocesar)

print("✅ Preprocesamiento completado.")
print("\nEjemplo de transformación:")
print(f"  ORIGINAL : {df['Consulta'].iloc[0][:100]}...")
print(f"  PROCESADO: {df['Consulta_Procesada'].iloc[0][:100]}...")


⚙️  Aplicando preprocesamiento (puede tardar unos minutos)...
✅ Preprocesamiento completado.

Ejemplo de transformación:
  ORIGINAL : Ayuda no se como inscribir asignaturas y nadie me explica....
  PROCESADO: ayuda inscribir asignatura explicar...


## 6. PREPARACIÓN DE FEATURES Y ETIQUETAS

In [18]:
df = df[df["Consulta_Procesada"].str.strip() != ""]
df = df.reset_index(drop=True)

X_raw = df["Consulta_Procesada"]
y_raw = df["Categoria_Mapeada"]

# Codificación de etiquetas
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)
NUM_CLASES = len(le.classes_)

# One-Hot Encoding para la red neuronal
y_onehot = keras.utils.to_categorical(y_encoded, num_classes=NUM_CLASES)

print(f"\n📦 Total de muestras: {len(X_raw)}")
print(f"🏷️  Número de clases : {NUM_CLASES}")
print(f"🏷️  Clases           : {list(le.classes_)}")


📦 Total de muestras: 35150
🏷️  Número de clases : 9
🏷️  Clases           : ['Actualización de datos personales', 'Calendario académico', 'Carnetización', 'Certificados', 'Gestión Académica', 'Gestión Económica', 'Información general sobre servicios estudiantiles', 'Política de gratuidad (matrícula cero) Pregrado', 'Reubicación socioeconómica en Pregrado']


# Semilla global para reproducibilidad

In [19]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("✅ Librerías cargadas correctamente.")
print(f"   TensorFlow version: {tf.__version__}")

✅ Librerías cargadas correctamente.
   TensorFlow version: 2.21.0


## 7. VECTORIZACIÓN TF-IDF

In [21]:
# NOTA: Las redes neuronales se benefician de vectores densos
# normalizados — usamos StandardScaler después del TF-IDF
from scipy.sparse import csr_matrix

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    norm="l2",
)

X_tfidf_sparse: csr_matrix = csr_matrix(tfidf.fit_transform(X_raw))
X_tfidf = X_tfidf_sparse.toarray()   # Dense para Keras

# Normalización con StandardScaler — mejora convergencia de la red
scaler     = StandardScaler()
X_scaled   = scaler.fit_transform(X_tfidf)

INPUT_DIM  = X_scaled.shape[1]
print(f"\n🔢 Dimensión de entrada (features): {INPUT_DIM}")


🔢 Dimensión de entrada (features): 5000


## 8. DIVISIÓN TRAIN / VALIDATION / TEST

In [22]:
# Split 70% train | 15% validación | 15% test
X_train_val, X_test, y_train_val, y_test, ye_train_val, ye_test = \
    train_test_split(
        X_scaled, y_onehot, y_encoded,
        test_size=0.15,
        random_state=SEED,
        stratify=y_encoded,
    )

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.176,           # 0.176 × 0.85 ≈ 0.15 del total
    random_state=SEED,
    stratify=np.argmax(y_train_val, axis=1),
)

print(f"\n📊 Train      : {X_train.shape[0]} muestras")
print(f"   Validación : {X_val.shape[0]} muestras")
print(f"   Test       : {X_test.shape[0]} muestras")


📊 Train      : 24618 muestras
   Validación : 5259 muestras
   Test       : 5273 muestras


## 9. ARQUITECTURA DE LA RED NEURONAL

In [23]:
def construir_modelo(input_dim: int, num_clases: int) -> keras.Model:
    """
    Construye y compila la red neuronal MLP con:
    - Capas Dense con activación ReLU
    - Batch Normalization para estabilizar el entrenamiento
    - Dropout progresivo para regularización
    - Regularización L2 para evitar overfitting
    """
    entradas = keras.Input(shape=(input_dim,), name="input_tfidf")

    # ── Capa 1: 512 neuronas ──────────────────────────────
    x = layers.Dense(
        512,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_1",
    )(entradas)
    x = layers.BatchNormalization(name="bn_1")(x)
    x = layers.Dropout(0.4, name="dropout_1")(x)

    # ── Capa 2: 256 neuronas ──────────────────────────────
    x = layers.Dense(
        256,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_2",
    )(x)
    x = layers.BatchNormalization(name="bn_2")(x)
    x = layers.Dropout(0.3, name="dropout_2")(x)

    # ── Capa 3: 128 neuronas ──────────────────────────────
    x = layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_3",
    )(x)
    x = layers.BatchNormalization(name="bn_3")(x)
    x = layers.Dropout(0.2, name="dropout_3")(x)

    # ── Capa 4: 64 neuronas ───────────────────────────────
    x = layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_4",
    )(x)
    x = layers.BatchNormalization(name="bn_4")(x)
    x = layers.Dropout(0.1, name="dropout_4")(x)

    # ── Capa de salida: softmax multiclase ────────────────
    salidas = layers.Dense(
        num_clases,
        activation="softmax",
        name="output",
    )(x)

    modelo = keras.Model(inputs=entradas, outputs=salidas, name="MLP_Consultas")

    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return modelo


modelo_nn = construir_modelo(INPUT_DIM, NUM_CLASES)

print("\n🧠 Arquitectura de la Red Neuronal:")
modelo_nn.summary()


🧠 Arquitectura de la Red Neuronal:


Model: "MLP_Consultas"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_tfidf (InputLayer)        │ (None, 5000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │     2,560,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_1 (BatchNormalization)       │ (None, 512)            │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_2 (BatchNormalization)       │ (None, 256)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_3 (BatchNormalization)       │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_4 (BatchNormalization)       │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 9)              │           585 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,737,417 (10.44 MB)

 Trainable params: 2,735,497 (10.44 MB)

 Non-trainable params: 1,920 (7.50 KB)

## 10. CALLBACKS

In [24]:
callbacks = [
    # Detiene el entrenamiento si val_loss no mejora en 15 épocas
    EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        verbose=1,
    ),
    # Reduce la tasa de aprendizaje si val_loss se estanca
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1,
    ),
    # Guarda el mejor modelo durante el entrenamiento
    ModelCheckpoint(
        filepath="mejor_modelo_nn.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]

## 11. ENTRENAMIENTO

In [26]:
print("\n🚀 Iniciando entrenamiento de la Red Neuronal...")

# Pesos de clase para manejar desbalance
from sklearn.utils.class_weight import compute_class_weight

clases_unicas  = np.unique(np.argmax(y_train, axis=1))
pesos_clase    = compute_class_weight(
    class_weight="balanced",
    classes=clases_unicas,
    y=np.argmax(y_train, axis=1),
)
dict_pesos = dict(zip(clases_unicas, pesos_clase))
print(f"\n⚖️  Pesos de clase aplicados: {dict_pesos}")

historia = modelo_nn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=dict_pesos,
    callbacks=callbacks,
    verbose="auto",
)

print("\n✅ Entrenamiento completado.")
print(f"   Épocas ejecutadas: {len(historia.history['loss'])}")


🚀 Iniciando entrenamiento de la Red Neuronal...

⚖️  Pesos de clase aplicados: {np.int64(0): np.float64(5.09373060211049), np.int64(1): np.float64(16.88477366255144), np.int64(2): np.float64(8.547916666666667), np.int64(3): np.float64(1.4706093189964158), np.int64(4): np.float64(0.3716485507246377), np.int64(5): np.float64(0.247990329404654), np.int64(6): np.float64(0.8288888888888889), np.int64(7): np.float64(195.38095238095238), np.int64(8): np.float64(78.15238095238095)}
Epoch 1/100
769/770 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.1966 - loss: 2.2285
Epoch 1: val_accuracy improved from None to 0.33581, saving model to mejor_modelo_nn.keras

Epoch 1: finished saving model to mejor_modelo_nn.keras
770/770 ━━━━━━━━━━━━━━━━━━━━ 20s 25ms/step - accuracy: 0.2273 - loss: 2.2215 - val_accuracy: 0.3358 - val_loss: 2.0867 - learning_rate: 0.0010
Epoch 2/100
768/770 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3284 - loss: 2.0461
Epoch 2: val_accuracy improved from 0.33581 to 0.5900

## 12. CURVAS DE APRENDIZAJE

In [ ]:
print("\n📈 Resumen de curvas de aprendizaje:")
epocas_ejecutadas = len(historia.history["loss"])

df_historia = pd.DataFrame({
    "Epoca":        range(1, epocas_ejecutadas + 1),
    "Loss_Train":   historia.history["loss"],
    "Loss_Val":     historia.history["val_loss"],
    "Acc_Train":    historia.history["accuracy"],
    "Acc_Val":      historia.history["val_accuracy"],
})

# Mostrar cada 5 épocas
print(df_historia[df_historia["Epoca"] % 5 == 0].to_string(index=False))

mejor_epoca = df_historia.loc[df_historia["Acc_Val"].idxmax()]
print(f"\n🏆 Mejor época: {int(mejor_epoca['Epoca'])}")
print(f"   Val Accuracy: {mejor_epoca['Acc_Val']:.4f}")
print(f"   Val Loss    : {mejor_epoca['Loss_Val']:.4f}")


📈 Resumen de curvas de aprendizaje:
 Epoca  Loss_Train  Loss_Val  Acc_Train  Acc_Val
     5    0.776927  1.333339   0.740434 0.691386
    10    0.781278  1.573881   0.818060 0.686442
    15    0.517003  1.711734   0.931189 0.698422

🏆 Mejor época: 14
   Val Accuracy: 0.6990
   Val Loss    : 1.6890


## 13. EVALUACIÓN EN TEST

In [28]:
print("\n🎯 Evaluando en conjunto de TEST...")
loss_test, acc_test = modelo_nn.evaluate(X_test, y_test, verbose="auto")
print(f"   Test Loss    : {loss_test:.4f}")
print(f"   Test Accuracy: {acc_test:.4f} ({acc_test*100:.2f}%)")

# Predicciones
y_pred_proba = modelo_nn.predict(X_test, verbose="auto")
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = np.argmax(y_test, axis=1)

print("\n📋 Reporte de Clasificación:")
print(classification_report(
    y_true, y_pred,
    target_names=le.classes_,
    zero_division=0,
))

print("\n🔲 Matriz de Confusión:")
cm     = confusion_matrix(y_true, y_pred)
cm_df  = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
print(cm_df)


🎯 Evaluando en conjunto de TEST...
165/165 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6827 - loss: 1.3427
   Test Loss    : 1.3427
   Test Accuracy: 0.6827 (68.27%)
165/165 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

📋 Reporte de Clasificación:
                                                   precision    recall  f1-score   support

                Actualización de datos personales       0.51      0.52      0.52       115
                             Calendario académico       0.14      0.29      0.19        35
                                    Carnetización       0.73      0.83      0.78        69
                                     Certificados       0.73      0.82      0.77       398
                                Gestión Académica       0.70      0.58      0.63      1576
                                Gestión Económica       0.84      0.77      0.80      2363
Información general sobre servicios estudiantiles       0.40      0.58      0.47       707
  Política de gratuidad (matríc

In [34]:
# ============================================================
# EXTRACCIÓN EXPLÍCITA DE MÉTRICAS - RED NEURONAL
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    precision_score,
    classification_report,
)
import pandas as pd
import numpy as np

# ── Predicciones sobre el conjunto de TEST ───────────────────
y_pred_proba = modelo_nn.predict(X_test, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = np.argmax(y_test,       axis=1)

# ── Métricas globales ────────────────────────────────────────
accuracy  = accuracy_score(y_true, y_pred)

recall_macro    = recall_score(y_true, y_pred, average="macro",    zero_division=0)
recall_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)

f1_macro        = f1_score(y_true, y_pred, average="macro",        zero_division=0)
f1_weighted     = f1_score(y_true, y_pred, average="weighted",     zero_division=0)

precision_macro    = precision_score(y_true, y_pred, average="macro",    zero_division=0)
precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)

print("\n" + "=" * 55)
print("        📊 MÉTRICAS GLOBALES - RED NEURONAL")
print("=" * 55)
print(f"  🎯 Accuracy                  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  🔁 Recall    (macro)         : {recall_macro:.4f}  ({recall_macro*100:.2f}%)")
print(f"  🔁 Recall    (weighted)      : {recall_weighted:.4f}  ({recall_weighted*100:.2f}%)")
print(f"  📐 F1-Score  (macro)         : {f1_macro:.4f}  ({f1_macro*100:.2f}%)")
print(f"  📐 F1-Score  (weighted)      : {f1_weighted:.4f}  ({f1_weighted*100:.2f}%)")
print(f"  🎯 Precision (macro)         : {precision_macro:.4f}  ({precision_macro*100:.2f}%)")
print(f"  🎯 Precision (weighted)      : {precision_weighted:.4f}  ({precision_weighted*100:.2f}%)")
print("=" * 55)

# ── Métricas por clase ───────────────────────────────────────
reporte = classification_report(
    y_true, y_pred,
    target_names=le.classes_,
    zero_division=0,
    output_dict=True,         # Retorna diccionario para convertir a DataFrame
)

df_reporte = pd.DataFrame(reporte).T.round(4)

# Separar clases del resumen
df_clases  = df_reporte.loc[le.classes_]
df_resumen = df_reporte.loc[["accuracy", "macro avg", "weighted avg"]]

print("\n📋 MÉTRICAS POR CLASE:")
print("=" * 85)
print(df_clases[["precision", "recall", "f1-score", "support"]].to_string())
print("=" * 85)

print("\n📊 RESUMEN GENERAL:")
print("=" * 85)
print(df_resumen[["precision", "recall", "f1-score", "support"]].to_string())
print("=" * 85)

# ── Guardar métricas en Excel ────────────────────────────────
with pd.ExcelWriter("metricas_red_neuronal.xlsx", engine="openpyxl") as writer:
    df_clases.to_excel(writer,  sheet_name="Metricas_por_clase")
    df_resumen.to_excel(writer, sheet_name="Resumen_global")

print("\n💾 Métricas guardadas en: metricas_red_neuronal.xlsx")



        📊 MÉTRICAS GLOBALES - RED NEURONAL
  🎯 Accuracy                  : 0.6827  (68.27%)
  🔁 Recall    (macro)         : 0.5184  (51.84%)
  🔁 Recall    (weighted)      : 0.6827  (68.27%)
  📐 F1-Score  (macro)         : 0.4736  (47.36%)
  📐 F1-Score  (weighted)      : 0.6938  (69.38%)
  🎯 Precision (macro)         : 0.4568  (45.68%)
  🎯 Precision (weighted)      : 0.7147  (71.47%)

📋 MÉTRICAS POR CLASE:
                                                   precision  recall  f1-score  support
Actualización de datos personales                     0.5128  0.5217    0.5172    115.0
Calendario académico                                  0.1370  0.2857    0.1852     35.0
Carnetización                                         0.7308  0.8261    0.7755     69.0
Certificados                                          0.7326  0.8191    0.7734    398.0
Gestión Académica                                     0.6986  0.5780    0.6326   1576.0
Gestión Económica                                     0.8373  

## 14. ANÁLISIS DE CONFIANZA POR CLASE

In [29]:
# Característica exclusiva de redes neuronales con softmax
print("\n🔍 Análisis de confianza promedio por clase predicha:")
print("=" * 55)

confianzas_max = np.max(y_pred_proba, axis=1)

for i, clase in enumerate(le.classes_):
    mask = y_pred == i
    if mask.sum() > 0:
        conf_media = confianzas_max[mask].mean()
        n_muestras = mask.sum()
        print(f"  {clase[:40]:<42} "
              f"conf={conf_media:.3f}  n={n_muestras}")

print("=" * 55)


🔍 Análisis de confianza promedio por clase predicha:
  Actualización de datos personales          conf=0.809  n=117
  Calendario académico                       conf=0.730  n=73
  Carnetización                              conf=0.903  n=78
  Certificados                               conf=0.870  n=445
  Gestión Académica                          conf=0.678  n=1304
  Gestión Económica                          conf=0.784  n=2182
  Información general sobre servicios estu   conf=0.655  n=1018
  Política de gratuidad (matrícula cero) P   conf=0.634  n=24
  Reubicación socioeconómica en Pregrado     conf=0.764  n=32


## 15. FUNCIÓN DE PREDICCIÓN

In [30]:
def predecir_categoria(texto_nuevo: str) -> dict:
    """
    Predice la categoría de una nueva consulta con la Red Neuronal.
    Retorna categoría predicha, confianza y distribución softmax completa.
    """
    texto_proc = preprocesar(texto_nuevo)

    # Conversión explícita a csr_matrix para que el tipado reconozca toarray()
    texto_tfidf_sparse = csr_matrix(tfidf.transform([texto_proc]))
    texto_tfidf = texto_tfidf_sparse.toarray()
    texto_scaled = scaler.transform(texto_tfidf)

    probabilidades   = modelo_nn.predict(texto_scaled, verbose="auto")[0]
    pred_idx         = np.argmax(probabilidades)
    pred_categoria   = le.inverse_transform([pred_idx])[0]
    confianza        = float(probabilidades[pred_idx])

    probs_dict = {
        clase: round(float(prob), 4)
        for clase, prob in zip(le.classes_, probabilidades)
    }
    probs_ordenadas = dict(
        sorted(probs_dict.items(), key=lambda x: x[1], reverse=True)
    )

    return {
        "categoria_predicha": pred_categoria,
        "confianza":          round(confianza, 4),
        "probabilidades":     probs_ordenadas,
    }

## 16. PRUEBA CON EJEMPLOS

In [31]:
ejemplos = [
    "No me aparece el recibo de pago en el SIA y necesito pagarlo urgente",
    "Quisiera saber cómo inscribir materias para este semestre",
    "Necesito tramitar mi carné universitario",
    "Soy estrato 2 y quiero saber si aplico para matrícula cero",
    "Solicito certificado de notas para trámite externo",
    "Quiero aplazar mi matrícula por motivos económicos",
    "Mi PBM quedó muy alto y no tengo recursos para pagar",
    "No me asignaron cita de inscripción de asignaturas",
    "Necesito actualizar mi dirección y datos de contacto",
    "No sé cuándo empiezan las clases este semestre",
]

print("\n🧪 PRUEBAS DE PREDICCIÓN:")
print("=" * 68)
for ejemplo in ejemplos:
    resultado = predecir_categoria(ejemplo)
    print(f"\n📝 Consulta   : {ejemplo}")
    print(f"🏷️  Predicción : {resultado['categoria_predicha']}")
    print(f"🔒 Confianza  : {resultado['confianza']*100:.1f}%")
    top3 = list(resultado["probabilidades"].items())[:3]
    print(f"📊 Top-3 probs: {top3}")
print("=" * 68)


🧪 PRUEBAS DE PREDICCIÓN:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step

📝 Consulta   : No me aparece el recibo de pago en el SIA y necesito pagarlo urgente
🏷️  Predicción : Gestión Económica
🔒 Confianza  : 74.1%
📊 Top-3 probs: [('Gestión Económica', 0.741), ('Gestión Académica', 0.1621), ('Información general sobre servicios estudiantiles', 0.0505)]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step

📝 Consulta   : Quisiera saber cómo inscribir materias para este semestre
🏷️  Predicción : Gestión Académica
🔒 Confianza  : 73.0%
📊 Top-3 probs: [('Gestión Académica', 0.7305), ('Información general sobre servicios estudiantiles', 0.2041), ('Calendario académico', 0.0487)]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step

📝 Consulta   : Necesito tramitar mi carné universitario
🏷️  Predicción : Carnetización
🔒 Confianza  : 99.8%
📊 Top-3 probs: [('Carnetización', 0.998), ('Información general sobre servicios estudiantiles', 0.0009), ('Gestión Académica', 0.0003)]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step

📝 Consulta   : Soy es

## 17. GUARDAR RESULTADOS Y MODELO

In [33]:
# Predicciones sobre todo el dataset
X_all_tfidf_sparse: csr_matrix = csr_matrix(tfidf.transform(X_raw))
X_all_scaled = scaler.transform(X_all_tfidf_sparse.toarray())
y_all_proba  = modelo_nn.predict(X_all_scaled, verbose= "auto")
y_all_pred   = np.argmax(y_all_proba, axis=1)

df_resultado = df[[
    "Consulta", "Categoria", "Categoria_Mapeada", "Consulta_Procesada"
]].copy()

df_resultado["Prediccion"] = le.inverse_transform(y_all_pred)
df_resultado["Correcto"]   = (
    df_resultado["Categoria_Mapeada"] == df_resultado["Prediccion"]
)
df_resultado["Confianza"]  = np.max(y_all_proba, axis=1).round(4)

df_resultado.to_excel("resultados_red_neuronal.xlsx", index=False)
df_historia.to_excel("historia_entrenamiento_nn.xlsx",  index=False)

# Guardar modelo final
modelo_nn.save("modelo_red_neuronal_final.keras")

print("\n💾 Archivos guardados:")
print("   → resultados_red_neuronal.xlsx")
print("   → historia_entrenamiento_nn.xlsx")
print("   → modelo_red_neuronal_final.keras")
print("   → mejor_modelo_nn.keras  (mejor época por val_accuracy)")

1099/1099 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step

💾 Archivos guardados:
   → resultados_red_neuronal.xlsx
   → historia_entrenamiento_nn.xlsx
   → modelo_red_neuronal_final.keras
   → mejor_modelo_nn.keras  (mejor época por val_accuracy)
